# AI Equity Analyst

*A multi-step LLM pipeline that turns a stock ticker into a one-page investment briefing.*

Give it a US ticker (e.g. `AAPL`) and it pulls **live fundamentals and recent news**, then
runs **three chained GPT calls** — a fundamental analyst, a news analyst, and a senior
analyst that synthesizes both — and streams out a structured report covering valuation,
risks, and a short / medium / long-term outlook.

> Built while following an LLM engineering course (Week 1 project). The core idea —
> **chaining multiple LLM calls into an agent-like workflow** — extends the course's
> "brochure generator" into a real, reusable finance tool.

*For informational and educational purposes only — not investment advice.*

## How it works — a 3-stage LLM chain

Every LLM call is stateless, so the design passes each stage's output into the next:

```
ticker
  │
├─▶ [1] Fundamental analyst   yfinance .info  (valuation, margins, growth, balance sheet)
  │        └─ notes: Strengths / Concerns / Metrics to watch
  │
├─▶ [2] News analyst          yfinance .news  (recent headlines + summaries)
  │        └─ notes: Tone / Key developments / Risks
  │
└─▶ [3] Senior analyst        combines notes [1] + [2]
           └─ final report: Snapshot · Fundamentals · News · Fundamentals vs News
              · Outlook by Horizon · Bottom Line   (streamed live)
```

Each stage has its own **system prompt** (role + rules + output format) and a **user
prompt** built from live data.

## Setup

1. Install dependencies: `pip install yfinance openai python-dotenv`
2. Create a `.env` file in the project root with your OpenAI key:
   ```
   OPENAI_API_KEY=sk-proj-...
   ```
3. Run the cells top to bottom.

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import yfinance as yf

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print("API key looks good")
else:
    print("There might be a problem with your API key — check your .env file")

MODEL = 'gpt-5-nano'
openai = OpenAI()

## Step 0 — Exploring the `yfinance` library

Before writing any analysis, I inspected what the library actually returns — using
`type()`, `dir()`, and `json.dumps(..., indent=2)` to understand the data structure.
`.info` is a dict of ~150 fundamental fields; `.news` is a list of recent articles.

In [ ]:
# What can this Ticker object do?
obj = yf.Ticker("AAPL")
print(type(obj))
print([m for m in dir(obj) if not m.startswith("_")])

In [ ]:
# View the full .info dictionary as a markdown table
info = yf.Ticker("AAPL").info

md = "| key | value |\n|---|---|\n"
for key, value in info.items():
    md += f"| {key} | {value} |\n"
display(Markdown(md))

In [ ]:
# Inspect the structure of the .news data
news = yf.Ticker("AAPL").news

print(type(news), "| count:", len(news))
print("keys:", news[0].keys())
print(json.dumps(news[0], indent=2, default=str))

## Step 1 — Fundamental analysis (from `.info`)

The first LLM plays a **fundamental analyst**. The user prompt dumps all available
fundamental data; the system prompt decides what to analyze and enforces a compact output
(Strengths / Concerns / Key metrics to watch).

In [ ]:
info_system_prompt = """
You are an experienced equity fundamental analyst.
You will be given key fundamental data for a single US-listed company.
Note: ratio metrics (margins, growth, ROE, yields) are decimal fractions, e.g. 0.27 = 27%.

Analyze the company across these dimensions:
- Business & scale: what it does, how large/established it is
- Valuation: expensive or cheap? Use P/E, forward P/E, P/B, P/S, PEG, EV/EBITDA and compare to what is typical
- Profitability: gross/operating/net margins, ROE, ROA
- Growth: revenue and earnings growth
- Financial health: cash vs debt, debt-to-equity, current/quick ratios, free cash flow
- Shareholder returns: dividend yield and payout ratio
- Price & momentum: where price sits vs 52-week range and moving averages
- Analyst view: consensus rating and mean target vs current price

Be objective and cite the numbers. Skip any metric that is missing.
Do NOT write a long essay. Output concise analyst notes as markdown bullets,
grouped under **Strengths**, **Concerns**, and **Key metrics to watch**. Keep under ~250 words.
This is one input to a later combined report, so prioritize substance over polish."""

In [ ]:
def get_info_user_prompt(ticker):
    info = yf.Ticker(ticker).info
    user_prompt = f"Here is all the available fundamental data for {ticker}:\n\n"
    for key, value in info.items():
        user_prompt += f"{key}: {value}\n"
    return user_prompt

In [ ]:
def fundamental_analyst(ticker):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": info_system_prompt},
            {"role": "user", "content": get_info_user_prompt(ticker)}
        ],
    )
    return response.choices[0].message.content

In [ ]:
display(Markdown(fundamental_analyst("AAPL")))

## Step 2 — News analysis (from `.news`)

The second LLM reads recent headlines and summaries, judges the overall tone, and flags
key developments and risks. It's explicitly told it only has headlines (not full articles),
so it doesn't over-interpret.

In [ ]:
news_system_prompt = """
You are a financial news analyst.
You will be given a list of recent news headlines (with short summaries and dates)
about a single US-listed company. You have only the headlines and summaries,
not the full articles, so do not over-interpret — base your read on what is given.

Your job:
- Judge the overall tone (positive / negative / mixed) and briefly explain why.
- Identify the 2-4 most market-relevant developments (earnings, guidance, products,
  regulation, legal, management changes, M&A, major analyst moves).
- Flag any notable risks or red flags.
- Ignore duplicates, generic market-wide commentary, and clickbait/ads.

Output concise analyst notes in markdown, grouped under **Overall tone**,
**Key developments**, and **Risks to watch**. Keep it under ~200 words.
This is one input to a later combined report, so prioritize substance over polish.
If no meaningful news is provided, clearly state that no relevant recent news was available.
"""

In [ ]:
def get_news_user_prompt(ticker):
    news = yf.Ticker(ticker).news or []
    user_prompt = f"Here are the recent news headlines for {ticker}:\n\n"
    for item in news:
        c = item["content"]
        title = c.get("title", "")
        summary = c.get("summary", "")
        date = c.get("pubDate", "")
        user_prompt += f"- [{date}] {title}\n  {summary}\n\n"
    return user_prompt

In [ ]:
def analyze_news(ticker):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": news_system_prompt},
            {"role": "user", "content": get_news_user_prompt(ticker)}
        ]
    )
    return response.choices[0].message.content

In [ ]:
display(Markdown(analyze_news("AAPL")))

## Step 3 — Combined report

The third LLM receives **both** sets of notes and synthesizes them — explicitly calling out
where fundamentals and news **agree or conflict**, then giving a short / medium / long-term
outlook. The output is streamed for a typewriter effect (like ChatGPT).

In [ ]:
report_system_prompt = """
You are a senior investment analyst writing a concise one-page briefing on a single
US-listed stock for a personal portfolio review.
You are given two sets of notes: (1) a fundamental analysis and (2) a recent-news analysis.

Synthesize them into a clear, well-organized markdown report (no code blocks):
- Explicitly point out where the fundamentals and the news AGREE or CONFLICT
  (e.g., strong numbers but negative news, or a cheap valuation confirmed by good news).
- Assess the stock over three horizons: Short-term (weeks-months),
  Medium-term (6-18 months), and Long-term (multi-year).
- Stay balanced: give both the bull case and the bear case.
- Do not invent any data or facts not present in the two notes.

Use exactly these sections:
## Snapshot
## Fundamentals
## Recent News
## Fundamentals vs News
## Outlook by Horizon
## Bottom Line

End with this line: *This report is for informational purposes only and is not investment advice.*
"""

In [ ]:
def get_report_user_prompt(ticker):
    fundamental_notes = fundamental_analyst(ticker)
    news_notes = analyze_news(ticker)
    user_prompt = f"""Stock: {ticker}

=== FUNDAMENTAL ANALYSIS ===
{fundamental_notes}

=== NEWS ANALYSIS ===
{news_notes}
"""
    return user_prompt

In [ ]:
def create_report(ticker):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": report_system_prompt},
            {"role": "user", "content": get_report_user_prompt(ticker)}
        ],
        stream=True
    )
    response = ""
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=handle.display_id)

In [ ]:
create_report("AAPL")

## Step 4 — Run it on a whole portfolio

Because all the logic lives in one function, generating briefings for an entire watchlist
is just a loop.

In [ ]:
portfolio = ["AAPL", "MSFT", "NVDA"]

for ticker in portfolio:
    display(Markdown(f"# {ticker}"))
    create_report(ticker)

## What I learned

- **LLMs are stateless** — "memory" is just passing prior context back in every call.
- **Chaining LLM calls** (extract  extract  synthesize) beats one giant prompt, and is
  the foundation of agentic workflows.
- **System vs user prompts**: fix the role and output format in the system prompt; feed
  changing data through the user prompt.
- **Reading a brand-new library from scratch** with `type()`, `dir()`, `help()`, and
  `json.dumps()` instead of relying on tutorials.
- **Structured, format-locked prompts** keep results consistent across many tickers.

*Tech: Python · yfinance · OpenAI API · Jupyter*

*This project is for informational and educational purposes only — not investment advice.*